# CAMINO quickstart

This notebook is a small, self-contained entry point for CAMINO. It creates a synthetic JWST-like defocused exposure, builds a local filter table, and exercises the current public CAMINO APIs without requiring external calibration files.

Before running it, create an environment and install the repository in editable mode:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e '.[dev]'
```


## Imports

`camino` depends on `dLux`, so importing both in the same notebook is the expected configuration.


In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

import camino
import dLux as dl


## Create a synthetic filter table and a synthetic defocused exposure

The example avoids CAMINO's optional default throughput source by writing a local `F212N.dat` file and passing `filters_dir=...` explicitly.


In [ ]:
def make_donut(shape=(64, 64), centre=(37.0, 26.0), radius=9.0, width=2.5):
    yy, xx = np.indices(shape).astype(float)
    r = np.hypot(yy - centre[0], xx - centre[1])
    return 1000.0 * np.exp(-0.5 * ((r - radius) / width) ** 2) + 5.0


def write_filter_table(directory: Path, name='F212N'):
    wl = np.linspace(21000.0, 21400.0, 64)
    centre = wl.mean()
    width = 0.25 * (wl.max() - wl.min())
    tp = np.exp(-0.5 * ((wl - centre) / width) ** 2)
    path = directory / f'{name}.dat'
    np.savetxt(path, np.column_stack([wl, tp]))
    return path


def write_defocus_file(path: Path):
    hdr = fits.Header()
    hdr['OBS_ID'] = 'V07464177001P0000003104'
    hdr['PUPIL'] = 'WLP8'
    hdr['FILTER'] = 'F212N'
    hdr['MJD-AVG'] = 60310.25

    sci = make_donut().astype(np.float32)
    err = np.full_like(sci, 2.0)

    fits.HDUList([
        fits.PrimaryHDU(header=hdr),
        fits.ImageHDU(sci, name='SCI'),
        fits.ImageHDU(err, name='ERR'),
    ]).writeto(path, overwrite=True)
    return path


workdir = Path(tempfile.mkdtemp(prefix='camino-quickstart-'))
filters_dir = workdir / 'filters'
filters_dir.mkdir()
filter_path = write_filter_table(filters_dir)
fits_path = write_defocus_file(workdir / 'synthetic_defocus.fits')

filter_path, fits_path


## Load the exposure and compute throughput bins


In [ ]:
fit = camino.SinglePointFilterFit(nwavels=4)
exposure = camino.exposure_from_defocus_file(str(fits_path), fit, crop=64)
wavelengths_m, weights = camino.calc_throughput('F212N', nwavels=4, filters_dir=filters_dir)

print('Exposure key:', exposure.filename)
print('Target:', exposure.target)
print('Filter:', exposure.filter)
print('MJD:', exposure.mjd)
print('Cutout shape:', exposure.data.shape)
print('Pupil tag:', camino.get_pupil(exposure))
print('dLux-backed source type:', type(fit.source).__name__)
print('Uses dLux PointSource:', isinstance(fit.source, dl.PointSource))
print('Wavelength bins (m):', np.asarray(wavelengths_m))
print('Weights:', np.asarray(weights))


## Inspect the synthetic exposure and throughput


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(np.asarray(exposure.data))
axes[0].set_title('SCI cutout')
axes[1].imshow(np.asarray(exposure.err))
axes[1].set_title('ERR cutout')
axes[2].imshow(np.asarray(exposure.bad), cmap='gray_r')
axes[2].set_title('Bad-pixel mask')
axes[3].plot(np.asarray(wavelengths_m) * 1e6, np.asarray(weights), marker='o')
axes[3].set_xlabel('Wavelength [$\mu$m]')
axes[3].set_ylabel('Weight')
axes[3].set_title('Filter bins')
for ax in axes[:3]:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()


## Where to go next

- Replace `fits_path` with a real JWST/NIRCam FITS file that contains `SCI` and `ERR` extensions.
- Replace `filters_dir` with your own directory of `{FILTER}.dat` files, or install the optional data provider CAMINO uses by default.
- If you also want to run the dLux introductory tutorial in the same environment, install its extra notebook dependencies with `python -m pip install optax matplotlib tqdm`.

For a fuller setup guide, see `docs/how_to_run.md` in the repository.
